### Load Dependencies

In [3]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from src.dataset import DSMShadeDataset
from src.model import ShadyModel

### Define Parameters

In [2]:
# Directory containing training data
data_dir = "/Users/luc/Geomatics/Thesis/Data"
# Set Acceleration Device
device = torch.device("mps:0" if torch.backends.mps.is_available() else "cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
# Hyperparameters
input_size = 0 # What is the resolution of the images?
epochs = 10
LAMBDA = 100
optimizer_learning_rate = 0.0002
# Todo experiment with not messing with this value? (defined in DCGAN paper apparently)
optimizer_momentum = 0.5
ngpu = 1
if device == torch.device("cpu"):
    ngpu = 0

mps:0


In [3]:
# Instantiate the custom dataset with transformations
custom_dataset = DSMShadeDataset(data_dir)

# Create a DataLoader for batching and parallel data loading
dataloader = DataLoader(custom_dataset, batch_size=32, shuffle=True)

#Todo Train versus test?

In [5]:
model = ShadyModel(ngpu, device)
model.setup_models()
# disc_loss = nn.BCELoss  # bce loss
# disc_optimizer = optim.Adam(model.parameters(), lr=optimizer_learning_rate, betas=(optimizer_momentum, 0.999))
gen_optimizer = optim.Adam(model.generator.parameters(), lr=optimizer_learning_rate, betas=(optimizer_momentum, 0.999))

### Training Loop

In [2]:
for epoch in range(epochs):
    for i, data in enumerate(dataloader, 0):
        ### FORWARD PASS ###
        y_hat = model.generator(data)

        ### LOSS CALCULATION ###
        loss = model.discriminator(y_hat, data)

        ### BACKPROPAGATION ###
        gen_optimizer.zero_grad()
        loss.backward()
        gen_optimizer.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Loss {loss.item():.4f}")


mps:0
